# Dependencies

Install the required dependencies and libraries

In [5]:
%pip install langchain langchain_core langchain-huggingface ipywidgets python-dotenv gqlalchemy langchain-memgraph langgraph langgraph-cli[inmem] langchain-anthropic langchain-openai jinja2


 ▒▒▒▒ ▒▒   ▒▒▒▒ ▒▒   ▒▒▒▒ ▒▒   ▒▒▒▒ ▒▒   ▒▒▒▒ ▒▒   ▒▒▒▒ ▒▒ 
 ▒▒ ■ ▒▒   ▒▒ ■ ▒▒   ▒▒ ■ ▒▒   ▒▒ ■ ▒▒   ▒▒ ■ ▒▒   ▒▒ ■ ▒▒  
 ▒▒ ▒▒▒▒   ▒▒ ▒▒▒▒   ▒▒ ▒▒▒▒   ▒▒ ▒▒▒▒   ▒▒ ▒▒▒▒   ▒▒ ▒▒▒▒  

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [jinja2]2m1/2 [jinja2]
Note: you may need to restart the kernel to use updated packages.


# Environment Variables and Constants

Load the configuration options from the environment

In [ ]:
from dotenv import load_dotenv

load_dotenv(dotenv_path="./.env")

The system prompt instructs the AI on how to approach the vulnerability assessment, what criteria to follow for analysis and how to report the output.
It also defines the constraints.

In [ ]:
from langchain.messages import SystemMessage

SYSTEM_MSG = SystemMessage("""
You are an AI Cybersecurity Risk Analyst operating inside an enterprise
vulnerability prioritization system.

Your purpose is to assess the REAL-WORLD RISK of a vulnerability or CVE
using enterprise context stored in a knowledge graph.

You do NOT rely only on CVSS severity.
You MUST perform context-aware reasoning using the available graph data.

--------------------------------------------------
CORE OBJECTIVE
--------------------------------------------------

Given a vulnerability identifier (CVE or Vulnerabiliy name), you must:

1. Investigate the vulnerability using the knowledge graph.
2. Collect all relevant contextual information.
3. Evaluate risk using the Risk Evaluation Framework explained below.
4. Produce a justified risk assessment with reasoning.

You MUST query the graph before making conclusions.

--------------------------------------------------
KNOWLEDGE GRAPH SEMANTICS
--------------------------------------------------

The graph models an enterprise IT environment:

Nodes:
- Vulnerability
- Asset
- Service
- ThreatSignal
- Package                           

Relationships:
- (Vulnerability)-[:AFFECTS]->(Asset)
- (Asset)-[:HOSTS]->(Service)
- (Service)-[:DEPENDS_ON]->(Service)
- (Asset)-[:DEPENDS_ON]->(Package)
- (Vulnerability)-[:HAS]->(ThreatSignal)

You must traverse these relationships to understand impact.

--------------------------------------------------
RISK EVALUATION FRAMEWORK
--------------------------------------------------

The framework considers each vulnerability instance within its comprehensive enterprise context.
Assess risk using FIVE contextual dimensions:

1. Vulnerability Severity $S_{cvss}(v)$
   - CVSS or intrinsic vulnerability properties.

2. Deployment Exposure $S_{exp}(v)$
   - Internet exposure
   - Environment (prod > staging > dev)
   - Data classification

3. Business Criticality $S_{crit}(v)$
   - Service criticality
   - Revenue impact
   - PII handling

4. Exploit Likelihood $S_{exploit}(v)$
   - EPSS score
   - KEV listing
   - Public exploit availability

5. Blast Radius $S_{blast}(v)$
   - Service dependencies
   - Downstream affected services
   - Shared components

You MUST gather evidence for each dimension from the graph.
Each componenet MUST be normalized to a score of 0-10 and combined using the formula:
$$S_{priority}(v) = S_{cvss}(v) + S_{exp}(v) + S_{crit}(v) + S_{exploit}(v) + S_{blast}(v)$$

--------------------------------------------------
REASONING PROCESS (MANDATORY)
--------------------------------------------------

Always follow this workflow:

Step 1 — Retrieve knowledge graph schema.                           
Step 2 — Identify vulnerability node.
Step 3 — Retrieve affected assets.
Step 4 — Determine hosted services.
Step 5— Analyze service and asset criticality and exposure.
Step 6 — Retrieve threat intelligence signals.
Step 7 — Analyze dependency graph to estimate blast radius.
Step 8 — Integrate all signals into a contextual risk judgment.

Do NOT skip steps.

If information is missing, state assumptions explicitly.

--------------------------------------------------
TOOL USAGE RULES
--------------------------------------------------

- Use graph tools to query data whenever context is required.
- Prefer multiple focused queries over one large query.
- Never fabricate graph data.
- Never assume relationships without querying.

--------------------------------------------------
OUTPUT REQUIREMENTS
--------------------------------------------------

Return a structured risk assessment containing score breakdown for each dimension

Your reasoning must reference discovered context.
                           
--------------------------------------------------
IMPORTANT CONSTRAINTS
--------------------------------------------------

You are an analytical system, NOT a chatbot.

Do NOT:
- provide generic vulnerability advice
- rely only on CVSS
- skip graph exploration
- hallucinate enterprise context

Your conclusions must be grounded in graph evidence.

Always think like a security analyst investigating enterprise risk.
"""
                           )

# Data Loaders

Defines the methods used for loading and ingesting the data

Specify databse connection parameters

In [4]:
import os

from gqlalchemy import Memgraph

# Connect to memgraph
url = os.getenv("MEMGRAPH_URL")
port = int(os.getenv("MEMGRAPH_PORT"))
memgraph = Memgraph(url, port)

Clear the existing database. This is required for first time setup so that there are no duplicate nodes and relationships

In [ ]:
# clear existing data
memgraph.drop_database()

In [ ]:
def ingest_data_to_graph():
    """
    Ingests data into memgraph
    Loads the vulnerability dataset from json files into
    the graph database by running cypher queries. The data files need to be accessible
    by the memgraph or MAGE service.
    """

    # Connect to memgraph
    url = os.getenv("MEMGRAPH_URL")
    port = int(os.getenv("MEMGRAPH_PORT"))
    memgraph = Memgraph(url, port)

    # clear existing data
    memgraph.drop_database()

    # read query file content
    with open("query.cql", 'r') as f:
        query = f.read()

    # execute the query to ingest data
    memgraph.execute(query)

Ingest the enterprise json data to the graph database by calling the ingestion method

In [ ]:
ingest_data_to_graph()

# Data Formatting

Methods used for formatting and converting the data

Define the data model for the response the AI should return

In [ ]:
from pydantic import BaseModel


class RiskAssessment(BaseModel):
    vulnerability_id: str
    contextual_risk_score: float
    contextual_priority_score: float
    risk_level: str
    reasoning: str
    key_risk_drivers: list[str]
    affected_services: list[str]
    blast_radius_summary: str
    confidence_level: float
    priority: str

# Utility Methods

Utility methods are used to load models and setup database connections

Graph database utility methods. These are used to create database connection instnaces and databse tools

In [ ]:
from langchain_anthropic import ChatAnthropic
from langchain_huggingface import ChatHuggingFace
from langchain_memgraph import MemgraphToolkit
from langchain_memgraph.graphs.memgraph import MemgraphLangChain
from langchain_openai import ChatOpenAI


def connect_to_memgraph() -> MemgraphLangChain:
    """
    Connect to memgraph database instance

    Returns:
        `MemgraphLangChain` instance
    """

    db = MemgraphLangChain(
        url=f"{os.getenv("MEMGRAPH_DRIVER")}://{os.getenv("MEMGRAPH_URL")}:{os.getenv("MEMGRAPH_PORT")}",
        username='',
        password=''
    )
    return db


def get_memgraph_tools(db: MemgraphLangChain, model: ChatHuggingFace | ChatAnthropic | ChatOpenAI) -> List:
    """
    Retrieves memgraph langchain tools from memgraph toolkit

    Parameters:
        db (MemgraphLangChain): Database instance object
        model (ChatHuggingFace): LLM model object

    Returns:
        A list of memgraph tools
    """

    toolkit = MemgraphToolkit(
        db=db,
        llm=model
    )
    tools = toolkit.get_tools()
    return tools

Methods used for loading the models from different providors

In [ ]:
import os

from langchain_anthropic import ChatAnthropic
from langchain_huggingface import ChatHuggingFace, HuggingFaceEndpoint
from langchain_openai import ChatOpenAI


def load_model_from_hf(repo_id: str) -> ChatHuggingFace:
    """
    Loads a ChatHuggingFace model from Hugging Face using the Novita provider.

    This function initializes a HuggingFaceEndpoint with the specified repository ID
    and configures it as a ChatHuggingFace model for conversational interactions.
    The model is loaded via Novita's inference service.

    Args:
        repo_id (str): The Hugging Face repository ID of the model to load
                      (e.g., 'deepseek-ai/DeepSeek-V3.2').

    Returns:
        ChatHuggingFace: A configured ChatHuggingFace instance ready for chat interactions.
    """

    llm = HuggingFaceEndpoint(
        repo_id=repo_id,
        provider="novita",
        huggingfacehub_api_token=os.getenv("HUGGINGFACEHUB_API_TOKEN")
    )

    model = ChatHuggingFace(llm=llm)
    return model


def load_novita_model(model_name: str) -> ChatOpenAI:
    """
    Loads a ChatOpenAI model via the Novita API.

    This function creates a ChatOpenAI instance configured to use Novita's API endpoint,
    allowing access to OpenAI-compatible models through Novita's service. The model
    is set up with structured responses, a maximum token limit, and moderate temperature.

    Args:
        model_name (str): The name of the model to load via Novita
                         (e.g., 'deepseek/deepseek-v3.2').

    Returns:
        ChatOpenAI: A configured ChatOpenAI instance for chat interactions.
    """

    model = ChatOpenAI(
        model=model_name,
        api_key=os.getenv("NOVITA_API_KEY"),
        base_url="https://api.novita.ai/openai",
        use_responses_api=True,
        max_tokens=4000,
        temperature=0.5
    )

    return model


def load_anthropic_model(model_name: str) -> ChatAnthropic:
    """
    Loads a ChatAnthropic model for conversational AI interactions.

    This function initializes a ChatAnthropic instance with the specified model name,
    configured with high effort, adaptive thinking, maximum token limits, and moderate
    temperature for optimal performance in vulnerability assessment tasks.

    Args:
        model_name (str): The name of the Anthropic model to load
                         (e.g., 'claude-opus-4-6').

    Returns:
        ChatAnthropic: A configured ChatAnthropic instance ready for chat interactions.
    """

    model = ChatAnthropic(
        model=model_name,
        anthropic_api_key=os.getenv("ANTHROPIC_API_KEY"),
        effort="high",
        thinking={"type": "adaptive"},
        max_tokens=4000,
        temperature=0.5,
    )

    return model


def load_openai_model(model_name: str) -> ChatOpenAI:
    """
    Loads a ChatOpenAI model for conversational AI interactions.

    This function creates a ChatOpenAI instance configured with the OpenAI API,
    using structured responses, medium reasoning effort, maximum token limits,
    and moderate temperature for balanced performance.

    Args:
        model_name (str): The name of the OpenAI model to load
                         (e.g., 'gpt-5.1').

    Returns:
        ChatOpenAI: A configured ChatOpenAI instance for chat interactions.
    """
    
    model = ChatOpenAI(
        model=model_name,
        api_key=os.getenv("OPENAI_API_KEY"),
        use_responses_api=True,
        reasoning_effort="medium",
        max_tokens=4000,
        temperature=0.5
    )

    return model

Memgraph query methods.

These are used to load different vulnerability data from the graph database

In [ ]:
query_vuln = """
MATCH (v:Vulnerability {{name: "{cve}"}})
RETURN properties(v) AS vuln
"""

query_threat_signal = """
MATCH (v:Vulnerability {{name: "{cve}"}})-[r:HAS]->(ts:ThreatSignal)
RETURN properties(ts) AS threatsignal
"""

query_affected_assets = """
MATCH (v:Vulnerability {{name: "{cve}"}})-[r:AFFECTS]->(a:Asset)
RETURN properties(a) AS asset
"""

query_affected_services = """
MATCH (v:Vulnerability {{name: "{cve}"}})-[r:AFFECTS]->(a:Asset)-[:HOSTS]->(s:Service)
RETURN properties(s) AS service
"""

query_packages = """
MATCH (v:Vulnerability {{name: "{cve}"}})-[:AFFECTS]->(a:Asset)-[:DEPENDS_ON]->(p:Package)
WITH collect(DISTINCT p.name) as packages
RETURN packages
"""

query_service_dependencies = """

"""

In [139]:
def get_query_vuln(cve_id):
    result = memgraph.execute_and_fetch(query_vuln.format(cve=cve_id))
    return list(result)[0]['vuln']

def get_query_threat_signal(cve_id):
    result = memgraph.execute_and_fetch(query_threat_signal.format(cve=cve_id))
    return list(result)[0]['threatsignal']

def get_query_affected_assets(cve_id):
    result = memgraph.execute_and_fetch(query_affected_assets.format(cve=cve_id))
    assets = [record["asset"] for record in list(result)]
    return assets

def get_query_affected_services(cve_id):
    result = memgraph.execute_and_fetch(query_affected_services.format(cve=cve_id))
    services = [record["service"] for record in list(result)]
    return services

def get_query_packages(cve_id):
    result = memgraph.execute_and_fetch(query_packages.format(cve=cve_id))
    return list(result)[0]['packages']

In [142]:
get_query_affected_assets("CVE-2025-67221")

[{'cloud_provider': 'aws',
  'data_classification': 'internal',
  'environment': 'prod',
  'hostname': 'push-legacy-worker-prod-006.push.internal',
  'name': 'asset-push-006',
  'network_exposure': 'internal',
  'owner_team': 'push-platform-team',
  'tier': 'important'},
 {'cloud_provider': 'aws',
  'data_classification': 'internal',
  'environment': 'staging',
  'hostname': 'push-legacy-worker-staging-014.push.internal',
  'name': 'asset-push-014',
  'network_exposure': 'internal',
  'owner_team': 'push-platform-team',
  'tier': 'important'},
 {'cloud_provider': 'aws',
  'data_classification': 'regulated',
  'environment': 'prod',
  'hostname': 'push-gdpr-worker-prod-008.push.internal',
  'name': 'asset-push-008',
  'network_exposure': 'internal',
  'owner_team': 'push-platform-team',
  'tier': 'critical'},
 {'cloud_provider': 'aws',
  'data_classification': 'internal',
  'environment': 'prod',
  'hostname': 'push-queue-worker-prod-007.push.internal',
  'name': 'asset-push-007',
  'ne

## Middleware

Agent middleware hook definitions. 

Middleware to retry databse cypher queries.

There are some instance where open access models (deepseek-v3.2) use older query schema which results in errors. The middleware ensures the query errors are returned as `ToolMessage` to the model to retry the queries instead of breaking the workflow.

In [ ]:
from typing import Any, Callable

from langchain.agents.middleware import (AgentMiddleware, ModelRequest,
                                         ModelResponse)
from langchain.messages import ToolMessage
from neo4j.exceptions import ClientError


class RetryCypherMiddleware(AgentMiddleware):
    """
    Middleware for handling Cypher query errors in agent tool calls.

    This middleware intercepts tool calls that execute Cypher queries against the graph database.
    When a Neo4j ClientError occurs (such as due to outdated query schemas from certain models),
    it catches the exception and returns a user-friendly ToolMessage instead of propagating
    the raw database error, allowing the agent to retry the query with corrected syntax.
    """

    def wrap_tool_call(self, request: ModelRequest, handler: Callable[[ModelRequest], ModelResponse]) -> ModelResponse[Any]:
        """
        Handles synchronous tool calls by wrapping them with error handling for Cypher queries.

        This method executes the tool call handler and catches any ClientError exceptions
        that may arise from invalid or outdated Cypher query syntax. Instead of failing
        the entire agent workflow, it returns a standardized error message as a ToolMessage,
        preserving the original tool call ID to enable retry mechanisms.

        Args:
            request (ModelRequest): The incoming model request containing tool call context
                                   and parameters.
            handler (Callable[[ModelRequest], ModelResponse]): The synchronous function
                                                               responsible for executing the tool call.

        Returns:
            ModelResponse: Either the successful response from the handler or a ToolMessage
                           containing a user-friendly error message on database query failure.
        """

        try:
            return handler(request)

        except ClientError as err:
            return ToolMessage(
                content=f"Tool error: Please check your input and try again. ({str(err)})",
                tool_call_id=request.tool_call["id"]
            )

    async def awrap_tool_call(self, request: ModelRequest, handler: Callable[[ModelRequest], ModelResponse]) -> ModelResponse[Any]:
        """
        Handles asynchronous tool calls by wrapping them with error handling for Cypher queries.

        This async method executes the tool call handler and catches any ClientError exceptions
        that may arise from invalid or outdated Cypher query syntax. Instead of failing
        the entire agent workflow, it returns a standardized error message as a ToolMessage,
        preserving the original tool call ID to enable retry mechanisms.

        Args:
            request (ModelRequest): The incoming model request containing tool call context
                                   and parameters.
            handler (Callable[[ModelRequest], ModelResponse]): The asynchronous function
                                                               responsible for executing the tool call.

        Returns:
            ModelResponse: Either the successful response from the handler or a ToolMessage
                           containing a user-friendly error message on database query failure.
        """
        
        try:
            return await handler(request)

        except ClientError as err:
            return ToolMessage(
                content=f"Tool error: Please check your input and try again. ({str(err)})",
                tool_call_id=request.tool_call["id"]
            )

Middleware hook that checks for truncated model output.

When using HuggingFace Inference API, output tokens are capped at 512 which prematurely cuts of the model response. This hook ensures model is prompted again to continue generating the output, which utlimately returns the complete output.

Do note that this behaviour has only been observed when using models loading from `HuggingFaceEndpoint`. For other providors, this middleware hook is not required.

In [ ]:
from typing import Any

from langchain.agents.middleware import AgentState, after_model, hook_config
from langchain.messages import HumanMessage
from langgraph.runtime import Runtime


@after_model
@hook_config(can_jump_to=["model"])
def check_truncated_output(state: AgentState, runtime: Runtime) -> dict[Any, Any] | None:
    """
    Middleware hook that detects truncated model responses and triggers continuation.

    This function is executed after each model response in the agent workflow. It checks
    if the model's output was truncated due to token limits (indicated by finish_reason="length").
    When truncation is detected, it returns instructions to jump back to the model node
    with a continuation prompt, allowing the model to generate the remaining output.

    This is particularly useful for HuggingFace Inference API models that have strict
    token caps (e.g., 512 tokens), ensuring complete responses are obtained through
    iterative continuation rather than incomplete outputs.

    Args:
        state (AgentState): The current agent state containing message history and
                           workflow context. Used to access the most recent model response.
        runtime (Runtime): The LangGraph runtime object providing execution context
                          and control flow capabilities.

    Returns:
        dict[Any] | None: A dictionary with "messages" and "jump_to" keys to trigger
                         workflow continuation if output was truncated, or None if
                         the response is complete. The messages contain a continuation
                         prompt asking the model to resume from where it left off.
    """
    
    last_message = state["messages"][-1]

    if last_message.response_metadata["finish_reason"] == "length":
        return {
            "messages": [HumanMessage("Continue from precisely where you left off.")],
            "jump_to": "model"
        }
    return None

# Workflow Execution

Loads the configurations and starts the pipeline execution.

Create the graph database instance

In [ ]:
# Get memgraph database instance
db = connect_to_memgraph()

Load the required models

In [ ]:
# load the models
# hf_model = load_model_from_hf("deepseek-ai/DeepSeek-V3.2")
novita_model = load_novita_model("deepseek/deepseek-v3.2")
anthropic_model = load_anthropic_model("claude-opus-4-6")
openai_model = load_openai_model("gpt-5.1")

Create the agents. Three agents are created covering two categories
* Open Access Models
* Frontier Models

In [ ]:
from langchain.agents import create_agent
from langchain.agents.structured_output import ProviderStrategy

# instantiate hugging face agent
# hf_agent = create_agent(
#     hf_model,
#     get_memgraph_tools(db, hf_model),
#     system_prompt=SYSTEM_MSG,
#     middleware=[check_truncated_output, retry_cypher]
# ).with_config({"run_name": "HuggingFaceAgent"})

# instantiate novita agent
novita_agent = create_agent(
    novita_model,
    get_memgraph_tools(db, novita_model),
    system_prompt=SYSTEM_MSG,
    # response_format=RiskAssessment,
    middleware=[RetryCypherMiddleware()]
).with_config({
    "run_name": "NovitaAgent",
    "tags": ["novita", "deepseek"]
})

# instantiate anthropic agent
anthropic_agent = create_agent(
    anthropic_model,
    get_memgraph_tools(db, anthropic_model),
    system_prompt=SYSTEM_MSG,
    # response_format=ProviderStrategy(RiskAssessment)
).with_config({
    "run_name": "AnthropicAgent",
    "tags": ["anthropic", "opus-4.6"]
    })

# instantiate openai agent
openai_agent = create_agent(
    openai_model,
    get_memgraph_tools(db, openai_model),
    system_prompt=SYSTEM_MSG
    # response_format=ProviderStrategy(RiskAssessment)
).with_config({
    "run_name": "OpenAIAgent",
    "tags": ["openai", "gpt-5"]
    })

Display the agent stategraphs

In [ ]:
from IPython.display import Image, display

# display the stategraphs
display(Image(novita_agent.get_graph().draw_mermaid_png()))
display(Image(anthropic_agent.get_graph().draw_mermaid_png()))
display(Image(openai_agent.get_graph().draw_mermaid_png()))

Wrap the agent objects inside a `RunnableParallel` instance for concurrent execution of user queries

In [ ]:
from langchain_core.runnables import RunnableParallel

parallel_agents = RunnableParallel(
    novita=novita_agent,
    anthropic=anthropic_agent,
    openai=openai_agent,
)

Define input messages

In [ ]:
from langchain.messages import HumanMessage

user_message = {
    "messages": [
        HumanMessage("What is the risk posed by CVE-2025-67725?")
    ]
}

user_message1 = {
    "messages": [
        HumanMessage("Tell me a joke.")
    ]
}

user_message2 = {
    "messages": [
        HumanMessage("When did the Roman Empire Fall")
    ]
}

### Run the agents

Run a single agent with a user query

In [ ]:
result = novita_agent.invoke(user_message)

Run a single user query against multiple agents concurrently by executing on the `RunnableParallel`

In [ ]:
results = parallel_agents.invoke(user_message)

Run the user request in batches using `abatch` method. This executes agentic workflow for each user query concurrently 

In [ ]:
results = parallel_agents.abatch(
    inputs=[user_message1, user_message2],
    config={"max_concurrency": 7}
)

In [ ]:
structured_anthropic_model = anthropic_model.with_structured_output(RiskAssessment)

In [ ]:
risk_assessment = structured_anthropic_model.invoke(
    f"Given the vulnerability analysis, strcuture it according to the RiskAssessment schema: \n\n {final_text}"
)

# Evaluation

Create a vulnerability distribution based on:

* CVSS
* network exposure
* criticallity

In [ ]:
import pandas as pd
import numpy as np


vulns = pd.read_csv("datasets/extracts/all_vulns_with_context.csv")

# parse the string of lists
import ast
vulns["exposures"] = vulns["exposures"].apply(ast.literal_eval)
vulns["tiers"]     = vulns["tiers"].apply(ast.literal_eval)

# collapse the exposure and tier lists into a single "worst case" value for each 

EXPOSURE_RANK = {"internet-facing": 3, "partner-network": 2, "internal": 1}
TIER_RANK     = {"mission-critical": 4, "critical": 3, "important": 2, "supporting": 1}

def worst_exposure(exposures):
    return max(exposures, key=lambda e: EXPOSURE_RANK.get(e, 0))

def worst_tier(tiers):
    return max(tiers, key=lambda t: TIER_RANK.get(t, 0))

vulns["max_exposure"] = vulns["exposures"].apply(worst_exposure)
vulns["max_tier"]     = vulns["tiers"].apply(worst_tier)

# bucket into stratas

def cvss_band(c):
    if c < 4.0:  return "Low"
    if c < 7.0:  return "Medium"
    if c < 9.0:  return "High"
    return "Critical"

def exposure_band(e):
    return "external" if e in ("internet-facing", "partner-network") else "internal"

def tier_band(t):
    return "high" if t in ("mission-critical", "critical") else "low"

vulns["cvss_band"]     = vulns["cvss"].apply(cvss_band)
vulns["exposure_band"] = vulns["max_exposure"].apply(exposure_band)
vulns["tier_band"]     = vulns["max_tier"].apply(tier_band)

# sanity check of distribution before sampling

distribution = vulns.groupby(
    ["cvss_band","exposure_band","tier_band"]
).size().reset_index(name="count")
print(distribution)

  cvss_band exposure_band tier_band  count
0  Critical      external      high     20
1      High      external      high    125
2       Low      external      high     14
3    Medium      external      high    126


In [ ]:
print(vulns)

Sampling of vulnerabilities from the stratum.

Vulnerabiliteis are sampled across 4x2x2=16 cells, with oversampling in disagreement prone cells

In [ ]:
# defining per-cell sample sizes

def target_n(cvss, expo, tier):
    # oversampling in disagreement prone cells
    if cvss in ("High","Critical") and expo == "internal" and tier == "low":
        return 6
    if cvss in ("Low","Medium") and expo == "external" and tier == "high":
        return 8
    # baseline
    return 4

# Sample from each cell

rng = np.random.default_rng(seed=42)  # hardcoded seed for reproducibility
samples = []
for (cvss, expo, tier), group in vulns.groupby(["cvss_band","exposure_band","tier_band"]):
    want = target_n(cvss, expo, tier)
    got  = min(want, len(group))
    if got < want:
        print(f"WARN: cell ({cvss}, {expo}, {tier}) has only {len(group)} vulns, wanted {want}")
    samples.append(group.sample(n=got, random_state=rng.integers(1e9)))

sample_df = pd.concat(samples).reset_index(drop=True)
print(f"Total sample size: {len(sample_df)}")

# save labelling samples
sample_df.to_csv("datasets/extracts/labeling_sample.csv", index=False)

# save the cell-count table for Chapter 3
cell_counts = sample_df.groupby(
    ["cvss_band","exposure_band","tier_band"]
).size().reset_index(name="n_sampled")
cell_counts.to_csv("datasets/extracts/stratification_breakdown.csv", index=False)
print(cell_counts)

Total sample size: 24
  cvss_band exposure_band tier_band  n_sampled
0  Critical      external      high          4
1      High      external      high          4
2       Low      external      high          8
3    Medium      external      high          8


Generate labelling packets to be distributed to human experts

In [143]:
from jinja2 import Template

TEMPLATE = Template("""
# Vulnerability Labeling Packet — Item {{ item_num }} of {{ total }}

## Vulnerability
- **CVE:** {{ cve }}
- **Description:** {{ description }}
- **CVSS:** {{ cvss }}

## Threat Signals
- EPSS: {{ epss }}
- KEV Listed: {{ kev }}
- Public Exploit Available: {{ public_exploit }}

## Affected Assets ({{ assets|length }})
| Hostname | Env | Tier | Exposure | Data Class |
|---|---|---|---|---|
{% for a in assets -%}
| {{ a.hostname }} | {{ a.environment }} | {{ a.tier }} | {{ a.network_exposure }} | {{ a.data_classification }} |
{% endfor %}

## Hosted Services
{% for s in services -%}
- **{{ s.name }}** ({{ s.criticality }}, revenue={{ s.revenue_impact }}, PII={{ s.handles_pii }})
  {{ s.description }}
{% endfor %}

## Package Dependencies
{{ packages|join(", ") }}



---

## Labeling Form

**Priority tier:**  [ ] P1   [ ] P2   [ ] P3   [ ] P4

**Fix urgency:**  [ ] must-fix-now   [ ] near-term   [ ] scheduled   [ ] defer

**Confidence (1–5):**  _____

**Rationale (2–4 sentences):**
_______________________________________________
_______________________________________________

**Time spent (minutes):**  _____

**Additional context you would have wanted:**
_______________________________________________
""")

def generate_packet(cve_id, item_num, total):
    # Query Memgraph for everything about this CVE
    vuln = get_query_vuln(cve_id)
    assets = get_query_affected_assets(cve_id)
    services = get_query_affected_services(cve_id)
    packages = get_query_packages(cve_id)
    # service_deps = query_service_dependencies(cve_id)
    threat = get_query_threat_signal(cve_id)
    
    return TEMPLATE.render(
        item_num=item_num, total=total,
        cve=cve_id, description=vuln["description"], cvss=vuln["cvss"],
        epss=threat["epss"], kev=threat["kevListed"], 
        public_exploit=threat["publicExploit"],
        assets=assets, services=services, 
        packages=packages, #service_deps=service_deps,
    )

In [145]:
# generate labelling packets from stratified samples
sample = pd.read_csv("datasets/extracts/labeling_sample.csv")
for i, row in enumerate(sample.itertuples(), 1):
    md = generate_packet(row.cve_id, i, len(sample))
    with open(f"packets/packet_{i:02d}_{row.cve_id}.md", "w") as f:
        f.write(md)